# Deepfake Detection Evaluation on FaceForensics++
## Re-running Xception Baseline & RGB+FFT Dual-Stream Models

This notebook clones the repository `https://github.com/Deeptrust-us/deepfake_image_video.git`, downloads FaceForensics++ (c23 compression), preprocesses spatial and FFT frequency domain representations, and evaluates both the **Xception baseline** and the **RGB+FFT Dual-Stream model** to obtain:
- **Accuracy**
- **F1-Score**
- **AUC (Area Under ROC Curve)**
- **EER (Equal Error Rate)**

All evaluation metrics use F1-optimal threshold selection on validation data.

### Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone repo into /content/deepfake_image_video
!git clone https://github.com/Deeptrust-us/deepfake_image_video.git /content/deepfake_image_video
%cd /content/deepfake_image_video

# Install requirements
!pip install -r /content/deepfake_image_video/requirements.txt
!apt-get update && apt-get install -y ffmpeg

### Step 2: Download FaceForensics++ Dataset (Deepfakes + Original)

In [ ]:
import os
os.environ['NONINTERACTIVE'] = '1'

# Download subset of Deepfakes and Original YouTube videos into /content/deepfake_image_video/data/faceforensics_raw
!python /content/deepfake_image_video/scripts/download/download_faceforensics.py /content/deepfake_image_video/data/faceforensics_raw -d Deepfakes -c c23 -t videos -n 50 --server EU2
!python /content/deepfake_image_video/scripts/download/download_faceforensics.py /content/deepfake_image_video/data/faceforensics_raw -d original -c c23 -t videos -n 50 --server EU2

### Step 3: Dataset Preprocessing (Frame Extraction, MTCNN Face Detection, FFT Maps)

In [ ]:
# Extract frames, faces, and FFT frequency spectrums into /content/deepfake_image_video/data/
!python /content/deepfake_image_video/scripts/preprocess.py --config /content/deepfake_image_video/config/config.yaml --dataset-type faceforensics --videos-dir /content/deepfake_image_video/data/faceforensics_raw --max_videos 100

### Step 4: Evaluate Xception Baseline Model

In [ ]:
!python /content/deepfake_image_video/scripts/evaluate.py --model xception --config /content/deepfake_image_video/config/config.yaml --split test --optimal_threshold --output_dir /content/deepfake_image_video/results

### Step 5: Evaluate RGB+FFT Dual-Stream Model

In [ ]:
!python /content/deepfake_image_video/scripts/evaluate.py --model rgb_fft_dual_stream --config /content/deepfake_image_video/config/config.yaml --split test --optimal_threshold --output_dir /content/deepfake_image_video/results

### Step 6: Summary Table & Log Packaging

In [ ]:
import os
import glob

print("="*60)
print("RESULTS SUMMARY (FaceForensics++ Deepfakes, c23)")
print("="*60)

result_files = glob.glob("/content/deepfake_image_video/results/*_results.txt")
for rf in result_files:
    print(f"\n--- {os.path.basename(rf)} ---")
    with open(rf, 'r') as f:
        print(f.read())

# Package results and logs
!tar -czf /content/deepfake_image_video/faceforensics_eval_results.tar.gz /content/deepfake_image_video/results/ /content/deepfake_image_video/logs/